# Low-rank Predicate Text Decomposition Analysis

This notebook isolates the current predicate prompt + low-rank decomposition module from the scene-graph model. It answers one question: can the low-rank module itself reconstruct and organize CLIP predicate text features well enough to be a useful relation classifier basis?

It does not load the detector, relation head, image crops, or SGG training loop.

## What To Look For

- `recon_cos_mean`: should be high. If it stays low, the rank or initialization is not enough.
- `offdiag_abs_mean`: lower means reconstructed predicate classifiers are less collapsed.
- `top1_match`: how often each reconstructed feature is closest to its original predicate feature.
- `W_active_mean`: average number of active basis components per predicate; lower means sparser address/composition.
- `base_to_novel_cos_gap`: whether novel predicates remain separable from base predicates after reconstruction.

A good decomposition usually has high reconstruction cosine, high top-1 self match, non-trivial sparsity, and no obvious collapse to frequent predicates.

In [ ]:
# Parameters
from pathlib import Path
import os
import torch

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'maskrcnn_benchmark').exists():
    PROJECT_ROOT = Path('/Users/shangfei/Developer/SDSGG')

PROMPT_JSON = os.environ.get(
    'PREDICATE_PROMPT_JSON',
    '/workspace/ccloud/sf/SDSGG/analysis/predicate_factor_descriptions.json',
)
PROMPT_FIELD = 'core_prompt'

RANKS = [4, 8, 16, 24, 32]
TRAIN_RANK = 16
STEPS = 1500
LR = 1e-3
TRAIN_BASIS = True
TRAIN_MODE = 'w'  # w: rel classifier path would update W only; decomposition losses still update B if TRAIN_BASIS=True
INIT_METHOD = 'semantic'  # use 'pca' for residual principal directions; use 'semantic' for predicate-text prototype bases

RECON_LOSS_WEIGHT = 1.0
SPARSITY_WEIGHT = 0.02
BASIS_DECORR_WEIGHT = 5e-4
WEIGHT_DECORR_WEIGHT = 2e-4
BASIS_ANCHOR_WEIGHT = 0.01
BASIS_SEMANTIC_WEIGHT = 0.05
SPARSITY_THRESHOLD = 0.05

DEVICE = os.environ.get('LOW_RANK_DEVICE', 'cuda' if torch.cuda.is_available() else 'cpu')
print('project:', PROJECT_ROOT)
print('prompt:', PROMPT_JSON)
print('device:', DEVICE)

In [ ]:
# Imports and repo modules
import sys
sys.path.insert(0, str(PROJECT_ROOT))

import json
import math
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

from CLIP import clip
from maskrcnn_benchmark.config import cfg
from maskrcnn_benchmark.modeling.roi_heads.relation_head.low_rank_text import (
    VG_PREDICATES,
    load_relation_prompt_texts,
    build_predicate_splits,
)
from maskrcnn_benchmark.modeling.roi_heads.relation_head.hola_low_rank import HOLaLowRankDecomposer

torch.set_grad_enabled(True)
torch.manual_seed(2026)
np.random.seed(2026)

In [ ]:
# Load current prompt text and encode with CLIP text encoder only
prompt_path = Path(PROMPT_JSON)
if not prompt_path.exists():
    candidates = [
        PROJECT_ROOT / 'analysis' / 'predicate_factor_descriptions.json',
        PROJECT_ROOT / 'predicate_factor_descriptions.json',
        Path('/workspace/ccloud/sf/SDSGG/analysis/predicate_factor_descriptions.json'),
    ]
    for candidate in candidates:
        if candidate.exists():
            prompt_path = candidate
            break
if not prompt_path.exists():
    raise FileNotFoundError(f'Prompt JSON not found. Set PROMPT_JSON or PREDICATE_PROMPT_JSON. Tried: {prompt_path}')

predicate_names = list(VG_PREDICATES)
fg_predicate_names = predicate_names[1:]
relation_texts = load_relation_prompt_texts(str(prompt_path), predicate_names, PROMPT_FIELD)
assert len(relation_texts) == len(fg_predicate_names)

clip_model, _ = clip.load('ViT-B/32', device=DEVICE)
clip_model.eval()
for p in clip_model.parameters():
    p.requires_grad = False

with torch.no_grad():
    tokens = clip.tokenize(relation_texts).to(DEVICE)
    text_features = clip_model.encode_text(tokens).float()
    text_features = F.normalize(text_features, dim=-1)

print('foreground predicates:', len(fg_predicate_names))
print('text feature shape:', tuple(text_features.shape))
pd.DataFrame({'predicate': fg_predicate_names, 'prompt': relation_texts}).head(10)

In [ ]:
# Split metadata: indices here are foreground-only, so global predicate id k maps to fg index k - 1.
splits_global = build_predicate_splits(cfg)
splits_fg = {
    name: torch.as_tensor(ids[1:], device=DEVICE, dtype=torch.long) - 1
    for name, ids in splits_global.items()
}
for name, ids in splits_fg.items():
    print(name, len(ids), [fg_predicate_names[i] for i in ids[:8].detach().cpu().tolist()])

In [ ]:
# Metric helpers
def pairwise_offdiag_abs(x):
    x = F.normalize(x.float(), dim=-1)
    sim = x @ x.t()
    mask = ~torch.eye(sim.size(0), dtype=torch.bool, device=sim.device)
    return sim[mask].abs().mean().item()

def reconstruction_metrics(original, reconstructed, weights, basis, threshold=0.05):
    original = F.normalize(original.float(), dim=-1)
    reconstructed = F.normalize(reconstructed.float(), dim=-1)
    cos = F.cosine_similarity(original, reconstructed, dim=-1)
    sim = reconstructed @ original.t()
    top1 = sim.argmax(dim=1)
    top5 = sim.topk(k=min(5, sim.size(1)), dim=1).indices
    eye = torch.arange(sim.size(0), device=sim.device)
    abs_w = weights.float().abs()
    active = (abs_w > threshold).float().sum(dim=1)
    max_share = abs_w.max(dim=1).values / abs_w.sum(dim=1).clamp_min(1e-6)
    basis_sim = F.normalize(basis.float(), dim=-1) @ F.normalize(basis.float(), dim=-1).t()
    basis_text_sim = F.normalize(basis.float(), dim=-1) @ F.normalize(original.float(), dim=-1).t()
    basis_mask = ~torch.eye(basis_sim.size(0), dtype=torch.bool, device=basis_sim.device)
    return {
        'recon_cos_mean': cos.mean().item(),
        'recon_cos_min': cos.min().item(),
        'recon_cos_std': cos.std().item(),
        'top1_match': (top1 == eye).float().mean().item(),
        'top5_match': (top5 == eye[:, None]).any(dim=1).float().mean().item(),
        'orig_offdiag_abs_mean': pairwise_offdiag_abs(original),
        'recon_offdiag_abs_mean': pairwise_offdiag_abs(reconstructed),
        'basis_offdiag_abs_mean': basis_sim[basis_mask].abs().mean().item(),
        'basis_text_max_cos_mean': basis_text_sim.max(dim=1).values.mean().item(),
        'basis_text_max_cos_min': basis_text_sim.max(dim=1).values.min().item(),
        'W_abs_mean': abs_w.mean().item(),
        'W_active_mean': active.mean().item(),
        'W_active_max': active.max().item(),
        'W_max_share_mean': max_share.mean().item(),
    }

def split_separation_metrics(features):
    features = F.normalize(features.float(), dim=-1)
    out = {}
    base = splits_fg['base']
    novel = splits_fg['novel']
    for split_name, ids in splits_fg.items():
        sim = features[ids] @ features[ids].t()
        mask = ~torch.eye(sim.size(0), dtype=torch.bool, device=sim.device)
        out[f'{split_name}_intra_abs'] = sim[mask].abs().mean().item() if mask.any() else 0.0
    base_novel = features[base] @ features[novel].t()
    out['base_to_novel_abs'] = base_novel.abs().mean().item()
    out['base_to_novel_max'] = base_novel.max(dim=0).values.mean().item()
    return out

def describe_nearest_neighbors(original, reconstructed, names, k=5):
    sim = F.normalize(reconstructed.float(), dim=-1) @ F.normalize(original.float(), dim=-1).t()
    rows = []
    for i, name in enumerate(names):
        vals, idxs = sim[i].topk(k)
        rows.append({
            'predicate': name,
            'self_cos': sim[i, i].item(),
            'top_neighbors': ', '.join(f'{names[j]}:{v:.3f}' for v, j in zip(vals.tolist(), idxs.tolist())),
            'top1_is_self': int(idxs[0].item() == i),
        })
    return pd.DataFrame(rows)

In [ ]:
# Rank sweep with the selected initialization, before gradient training.
rank_rows = []
for rank in RANKS:
    model = HOLaLowRankDecomposer(
        text_features,
        rank=rank,
        init_method=INIT_METHOD,
        train_basis=TRAIN_BASIS,
        train_mode=TRAIN_MODE,
        recon_loss_weight=RECON_LOSS_WEIGHT,
        sparsity_weight=SPARSITY_WEIGHT,
        basis_decorr_weight=BASIS_DECORR_WEIGHT,
        weight_decorr_weight=WEIGHT_DECORR_WEIGHT,
        basis_anchor_weight=BASIS_ANCHOR_WEIGHT,
        basis_semantic_weight=BASIS_SEMANTIC_WEIGHT,
    ).to(DEVICE)
    with torch.no_grad():
        rec = model.reconstruct()
        row = {'rank': rank, **reconstruction_metrics(text_features, rec, model.class_weights, model.basis_feat, SPARSITY_THRESHOLD)}
        row.update(split_separation_metrics(rec))
        rank_rows.append(row)

rank_df = pd.DataFrame(rank_rows)
display(rank_df.round(4))
rank_df.plot(x='rank', y=['recon_cos_mean', 'top1_match', 'basis_offdiag_abs_mean'], marker='o', figsize=(8, 4), grid=True)
plt.title('PCA initialization quality by rank')
plt.show()

In [ ]:
# Train only the decomposition module. No image model and no relation classifier loss.
decomposer = HOLaLowRankDecomposer(
    text_features,
    rank=TRAIN_RANK,
    init_method=INIT_METHOD,
    train_basis=TRAIN_BASIS,
    train_mode=TRAIN_MODE,
    recon_loss_weight=RECON_LOSS_WEIGHT,
    sparsity_weight=SPARSITY_WEIGHT,
    basis_decorr_weight=BASIS_DECORR_WEIGHT,
    weight_decorr_weight=WEIGHT_DECORR_WEIGHT,
    basis_anchor_weight=BASIS_ANCHOR_WEIGHT,
    basis_semantic_weight=BASIS_SEMANTIC_WEIGHT,
).to(DEVICE)

optimizer = torch.optim.AdamW([p for p in decomposer.parameters() if p.requires_grad], lr=LR, weight_decay=0.0)
history = []

def eval_decomposer(step):
    with torch.no_grad():
        rec = decomposer.reconstruct()
        metrics = reconstruction_metrics(text_features, rec, decomposer.class_weights, decomposer.basis_feat, SPARSITY_THRESHOLD)
        metrics.update(split_separation_metrics(rec))
        metrics['step'] = step
        return metrics

history.append(eval_decomposer(0))
for step in range(1, STEPS + 1):
    optimizer.zero_grad()
    losses = decomposer.losses()
    total = sum(losses.values())
    total.backward()
    torch.nn.utils.clip_grad_norm_(decomposer.parameters(), 5.0)
    optimizer.step()
    if step % 50 == 0 or step == 1:
        row = eval_decomposer(step)
        row.update({k: v.detach().float().item() for k, v in losses.items()})
        row['total_loss'] = total.detach().float().item()
        history.append(row)

hist_df = pd.DataFrame(history)
display(hist_df.tail().round(4))

In [ ]:
# Training curves
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
hist_df.plot(x='step', y=['recon_cos_mean', 'recon_cos_min', 'top1_match'], ax=axes[0], grid=True)
axes[0].set_title('Reconstruction quality')
hist_df.plot(x='step', y=['W_active_mean', 'W_max_share_mean'], ax=axes[1], grid=True)
axes[1].set_title('Weight sparsity / concentration')
hist_df.plot(x='step', y=['basis_offdiag_abs_mean', 'recon_offdiag_abs_mean', 'basis_text_max_cos_mean'], ax=axes[2], grid=True)
axes[2].set_title('Decorrelation / collapse checks')
plt.tight_layout()
plt.show()

In [ ]:
# Per-predicate reconstruction report
with torch.no_grad():
    reconstructed = decomposer.reconstruct()
    report = describe_nearest_neighbors(text_features, reconstructed, fg_predicate_names, k=5)

print('Worst reconstructed predicates:')
display(report.sort_values('self_cos').head(12).round(4))
print('Potential collapsed predicates, where nearest original is not itself:')
display(report[report['top1_is_self'] == 0].sort_values('self_cos').round(4))

In [ ]:
# Inspect basis-address weights W. Sparse and interpretable addresses should show a few dominant basis components per predicate.
with torch.no_grad():
    W = decomposer.class_weights.detach().float().cpu().numpy()
    absW = np.abs(W)
    top_components = []
    for i, name in enumerate(fg_predicate_names):
        idx = absW[i].argsort()[::-1][:5]
        top_components.append({
            'predicate': name,
            'l1': absW[i].sum(),
            'active': int((absW[i] > SPARSITY_THRESHOLD).sum()),
            'top_basis': ', '.join(f'b{j}:{W[i, j]:.3f}' for j in idx),
        })
W_report = pd.DataFrame(top_components)
display(W_report.sort_values('active').head(10))
display(W_report.sort_values('active', ascending=False).head(10))

In [ ]:
# Base / novel diagnostic. This helps test whether the compact basis keeps novel predicate text features usable.
with torch.no_grad():
    rec = F.normalize(decomposer.reconstruct().float(), dim=-1)
    ori = F.normalize(text_features.float(), dim=-1)
    rows = []
    for split_name, ids in splits_fg.items():
        cos = F.cosine_similarity(ori[ids], rec[ids], dim=-1)
        rows.append({
            'split': split_name,
            'count': len(ids),
            'recon_cos_mean': cos.mean().item(),
            'recon_cos_min': cos.min().item(),
            'recon_cos_std': cos.std().item() if len(ids) > 1 else 0.0,
        })
split_df = pd.DataFrame(rows)
display(split_df.round(4))

base = splits_fg['base']
novel = splits_fg['novel']
sim_bn = rec[novel] @ rec[base].t()
nearest_base = sim_bn.argmax(dim=1).detach().cpu().tolist()
novel_rows = []
for local_i, base_j in enumerate(nearest_base):
    novel_idx = novel[local_i].item()
    base_idx = base[base_j].item()
    novel_rows.append({
        'novel_predicate': fg_predicate_names[novel_idx],
        'nearest_base': fg_predicate_names[base_idx],
        'cos': sim_bn[local_i, base_j].item(),
    })
display(pd.DataFrame(novel_rows).sort_values('cos', ascending=False).round(4))

## How To Judge Effectiveness

Use these rough thresholds as a sanity check, not as paper claims:

- If `top1_match` is below `0.8`, the decomposed classifier no longer preserves predicate identity well.
- If `recon_cos_min` is very low, inspect the worst predicates; these are likely where relation training will collapse.
- If `W_active_mean` is close to `rank`, the module is not really sparse/address-like.
- If `basis_offdiag_abs_mean` remains high, the basis vectors are not disentangled.
- If novel predicates all map to the same nearest base predicate, the basis is compact but not useful for open-vocabulary transfer.

Recommended next experiment: run this notebook with `TRAIN_BASIS=False` and compare it with `TRAIN_BASIS=True`. If fixed PCA basis already has high reconstruction and better stability, use fixed basis for the first SGG training stage.